![hslu_logo.png](img/hslu_logo.png)

## Week 5

<hr style="border:1px solid black">


# Excercise: Live processing of SSD (single shot detector)
---
---
This excercise is to illustrate the live processing of an object detector called SSD.

### Import necessary packages

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time

from utils import plot_img

#### List all available models

In [ ]:
all_models = torchvision.models.list_models()
for mod in range(len(all_models)):
    print(all_models[mod])

#### Select the model
We use ssd due to its yet simple architecture

In [ ]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

In [ ]:
#architecture is more complex to provide detection capabilities
weights = SSD300_VGG16_Weights.DEFAULT
ssd300_vgg16_model = ssd300_vgg16(weights=weights)
ssd300_vgg16_model.eval()

#### Show the number of parameters

In [ ]:
tot_param = []
cum_param = 0
for name, parameter in ssd300_vgg16_model.named_parameters():
    print(name, parameter.numel())
    cum_param += parameter.numel()
    tot_param = tot_param + [cum_param]

plt.plot(tot_param, 'bo-')
plt.show()

#### Load an image

Note the shape with the channels at the first position

In [ ]:
img_name = '../SW03/sample_img/cat.jpg'
img = torchvision.io.read_image(img_name)
print(img.shape)
plot_img(torch.movedim(img, [0],[2]))

#### Transform the image

*Before using the pre-trained models, one must preprocess the image (resize with right resolution/interpolation, apply inference transforms, rescale the values etc). There is no standard way to do this as it depends on how a given model was trained. It can vary across model families, variants or even weight versions. Using the correct preprocessing method is critical and failing to do so may lead to decreased accuracy or incorrect outputs.*
[[ref](https://pytorch.org/vision/stable/models.html#using-the-pre-trained-models)]

The preprocessing is weight specific (a model can have different weight configurations) and therefore the weights (through the method `transforms()`) acutally contain the information on the precise transformation required. 


In [ ]:
img_trans = weights.transforms()(img)
print('\noriginal shape: ', img.shape)
print('transform shape: ', img_trans.shape)
print('\noriginal type: ', img.dtype)
print('transform type: ', img_trans.dtype)
print('\noriginal range: [', torch.min(img).item(), ',', torch.max(img).item(), ']')
print('transform range: [', torch.min(img_trans).item(), ',', torch.max(img_trans).item(), ']')

#### Classify the image


In [ ]:
#we have to add the batch dimension (only single image)
batch = img_trans.unsqueeze(0)
print(batch.shape)

#apply the model (only used first batch result, [0])
prediction = ssd300_vgg16_model(batch)[0]

#result is a dict consisting of keys 'boxes', 'scores', 'labels'
print('prediction keys:', prediction.keys())

#results are ordered, we only take the highest score
object_id = 0

#get score and class
score = prediction['scores'][object_id]
category_name = weights.meta["categories"][prediction['labels'][object_id]]

print(f'\nscore: {100*score:.1f},  category: {category_name}')

#### Plot the bounding box

We use opencv for its simple use of `rectangle`

In [ ]:
#torch tensor makes problems -> convert to numpy
box = prediction['boxes'][object_id].detach().numpy().astype(np.int32)

img_2_plot = cv2.imread(img_name)
#opencv uses BGR as default
img_2_plot = cv2.cvtColor(img_2_plot, cv2.COLOR_BGR2RGB)
cv2.rectangle(img_2_plot, (box[:2]), (box[2:4]), color=(255,0,0),thickness=2)

plot_img(img_2_plot, figure_size=[5, 5])

### Setup Live Processing

We plot the results directly in the camera image

In [ ]:
#list of colors for different object ids
colors=[(255,0,0), (0,255,0), (0,0,255), (255,255,0), (255,0,255), (0,255,255)]
# font
font = cv2.FONT_HERSHEY_SIMPLEX
# fontScale
fontScale = .6
# line thickness
thickness = 1
#define a limit score to show object
limit_score = 0.1

# Open the default camera
#to check available capture devices best is matlab :-/
capture_device = 0

cam = cv2.VideoCapture(capture_device)


while True:
    start=time.time()
    ret, frame = cam.read()

    #image conversion 
    image = torch.zeros((3,frame.shape[0],frame.shape[1]),dtype=torch.uint8)
    for plane in range(3):
        image[plane] = torch.tensor(frame[:,:,2-plane])

    img_trans = weights.transforms()(image)
    prediction = ssd300_vgg16_model(img_trans.unsqueeze(0))[0]

    stop=time.time()

    img_2_plot = frame.copy()

    class_ind = 0
    score = prediction['scores'][class_ind]
    while (limit_score < score) and (class_ind < 6):#maximum of six objects (colors!)
        #draw rectangle
        box = prediction['boxes'][class_ind].detach().numpy().astype(np.int32)
        cv2.rectangle(img_2_plot, (box[:2]), (box[2:4]), colors[class_ind],thickness)
        #put category and score at up left corner
        category = weights.meta["categories"][prediction['labels'][class_ind].item()]
        cv2.putText(img_2_plot, f'{category}, score={100*score:.1f}', (box[:2] + [20,20]), font, 
                       fontScale, colors[class_ind], thickness)
        
        class_ind += 1
        score = prediction['scores'][class_ind]
    
    cv2.putText(img_2_plot, f'processing time {stop-start:.2f}', (20,20), font, 
                       fontScale, (255,255,255), thickness)

    # Display the captured frame
    cv2.imshow('Camera', img_2_plot)

    key = cv2.waitKeyEx(1)
    if key != -1:
        if key == 101: #'e'
            break
        
# Release the capture and writer objects
cam.release()
cv2.destroyWindow('Camera')